### embedding

In [23]:
import pandas as pd
import yaml
import re
import time

print("🚀 啟動階段二：動態地名脫敏 (YAML 配置版)")

# ==========================================
# 1. 載入 YAML 設定
# ==========================================
def load_sanitization_config(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    
    # 提取所有地名分類並合併成一個清單
    loc_config = config.get('locations', {})
    all_locations = []
    for category in loc_config:
        all_locations.extend(loc_config[category])
    
    # 將清單組合成 Regex 格式: (地名1|地名2|...)
    pattern_str = f"({'|'.join(all_locations)})"
    return re.compile(pattern_str)

# 載入並編譯
city_pattern = load_sanitization_config('keywords.yaml')
print(f"✅ 已從 YAML 載入 {city_pattern.groups} 個地名特徵進行防禦。")

# ==========================================
# 2. 脫敏處理函數 (支援 List)
# ==========================================
def preprocess_texts(texts, replacement="[LOC]"):
    if isinstance(texts, str):
        return city_pattern.sub(replacement, texts)
    return [city_pattern.sub(replacement, str(t)) if pd.notna(t) else "" for t in texts]

# ==========================================
# 3. 執行資料清洗
# ==========================================
print("載入 candidate_pool.csv 並執行脫敏...")
df = pd.read_csv('candidate_pool.csv')
df = df.dropna(subset=['name']).drop_duplicates(subset=['name']).reset_index(drop=True)

start_time = time.time()
df['original_name'] = df['name']
df['name'] = preprocess_texts(df['name'].tolist())

print(f"✅ 脫敏完成，耗時: {time.time() - start_time:.2f} 秒")

# 預覽效果
sample_hits = df[df['original_name'].str.contains(city_pattern, regex=True)].head(5)
for _, row in sample_hits.iterrows():
    print(f"原始: {row['original_name']} -> 清洗: {row['name']}")

# 儲存結果供 Step 3 使用
df.to_csv('candidate_pool_sanitized.csv', index=False, encoding='utf-8-sig')
print("\n💾 預處理後的資料已儲存為 candidate_pool_sanitized.csv")

🚀 啟動階段二：動態地名脫敏 (YAML 配置版)
✅ 已從 YAML 載入 1 個地名特徵進行防禦。
載入 candidate_pool.csv 並執行脫敏...
✅ 脫敏完成，耗時: 0.00 秒
原始: 高雄麗尊酒店 The Lees Hotel, Kaohsiung -> 清洗: [LOC]麗尊酒店 The Lees Hotel, Kaohsiung
原始: 翰品酒店 高雄 Chateau de Chine Kaohsiung -> 清洗: 翰品酒店 [LOC] Chateau de Chine Kaohsiung
原始: 台中金典酒店 -> 清洗: [LOC]金典酒店
原始: 台中金典酒店 婚禮企劃 -> 清洗: [LOC]金典酒店 婚禮企劃
原始: Mandarin Oriental, Taipei 台北文華東方酒店 -> 清洗: Mandarin Oriental, Taipei [LOC]文華東方酒店

💾 預處理後的資料已儲存為 candidate_pool_sanitized.csv


/var/folders/_r/lwhdpchj7m5cftwpvm8_4bgw0000gn/T/ipykernel_74978/4152041456.py:51: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



In [24]:
import pandas as pd

# 讀取候選池資料
df_candidates = pd.read_csv('candidate_pool_sanitized.csv', low_memory=False)

# 確保文字欄位沒有 NaN，並去除完全重複的文本
df_candidates = df_candidates.dropna(subset=['name'])
df_candidates = df_candidates.drop_duplicates(subset=['name']).reset_index(drop=True)

print(f"去重後的獨立候選樣本數: {len(df_candidates)} 筆")
df_candidates.head()

# --- 各類別關鍵字初步匹配統計 ---
matched_cols = [col for col in df_candidates.columns if col.startswith('matched_')]
true_counts = df_candidates[matched_cols].sum()
print(true_counts)

去重後的獨立候選樣本數: 6305 筆
matched_loan                2100
matched_porn                3191
matched_gambling             861
matched_scam_recruitment     166
dtype: int64


In [25]:
from sentence_transformers import SentenceTransformer
import torch
import time

# 確認 Apple Silicon MPS 加速是否啟動
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"使用的運算裝置: {device}")

# 載入輕量級多語系模型 (第一次執行會下載約 400MB 的模型檔)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=device)

# 提取要轉換的文字清單
texts = df_candidates['name'].tolist()

print("🚀 開始生成 Embeddings... ")
start_time = time.time()

# 轉換向量 (batch_size 設為 512，M3 Max 處理四千筆應該不到 10 秒)
embeddings = model.encode(texts, batch_size=512, show_progress_bar=True)

print(f"✅ 向量化完成！耗時: {time.time() - start_time:.2f} 秒")
print(f"向量矩陣維度: {embeddings.shape}")

使用的運算裝置: mps
🚀 開始生成 Embeddings... 


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

✅ 向量化完成！耗時: 1.53 秒
向量矩陣維度: (6305, 384)


In [26]:
import umap
import plotly.express as px

print("正在進行 UMAP 降維... (約需 10~30 秒)")

# 使用 UMAP 將向量降至  維
# n_neighbors 決定關注局部還是全域結構，15 是個好起點
reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=42, metric='cosine')
embeddings_2d = reducer.fit_transform(embeddings)

# 將座標合併回 DataFrame
df_candidates['x'] = embeddings_2d[:, 0]
df_candidates['y'] = embeddings_2d[:, 1]
df_candidates['z'] = embeddings_2d[:, 2] # 3d

# 決定該點顯示的顏色分類 (依據第一階段的標籤)
def get_color_category(row):
    if row.get('matched_loan', False): return '借貸融資'
    if row.get('matched_porn', False): return '黃特種行業'
    if row.get('matched_gambling', False): return '博弈'
    if row.get('matched_scam_recruitment', False): return '詐騙高風險招募'
    return '多重觸發'

df_candidates['color_category'] = df_candidates.apply(get_color_category, axis=1)

# 繪製 Plotly 互動式散佈圖
fig = px.scatter_3d(
    df_candidates, 
    x='x', y='y', z='z',
    color='color_category',
    hover_data=['name'], 
    title='意圖特徵空間流形分佈 (3D UMAP Projection)',
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.G10,
    width=1000, height=800 # 3D 圖建議稍微調大一點以便觀察
)

fig.update_traces(marker=dict(size=3))

# 隱藏座標軸數字 (因為 UMAP 的絕對座標沒有意義，相對距離才有意義)
fig.update_xaxes(showticklabels=False, title='')
fig.update_yaxes(showticklabels=False, title='')
fig.update_layout(width=1000, height=800, template="plotly_white")

fig.show()

正在進行 UMAP 降維... (約需 10~30 秒)


/opt/miniconda3/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [27]:
### k-means and centroid

In [28]:
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
import numpy as np

print("啟動黃金樣本自動採樣 (K-Means Centroid Sampling)...")

# 準備一個空的 DataFrame 來裝我們的黃金樣本候選
gold_candidates_list = []

# 我們要針對剛剛分類出來的每個類別進行採樣
categories = df_candidates['color_category'].unique()

for cat in categories:
    if cat == '多重觸發':
        continue # 先跳過邊界模糊的資料
        
    print(f"處理類別: {cat}")
    
    # 取出該類別的所有資料與對應的 Embedding
    cat_df = df_candidates[df_candidates['color_category'] == cat].copy()
    cat_indices = cat_df.index.tolist()
    cat_embeddings = embeddings[cat_indices]
    
    # 決定要分幾個小群 (Cluster)。如果是 50 筆樣本，我們切 x0 群，每群取前 5 近的點
    n_clusters = min(30, len(cat_df)) # 確保資料夠多
    
    if n_clusters == 0:
        continue
        
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(cat_embeddings)
    
    # 找出每個樣本距離其所屬群集中心的距離
    distances = kmeans.transform(cat_embeddings)
    
    # 針對每一個小群集，挑選距離中心最近的 5 筆資料
    for i in range(n_clusters):
        # 屬於第 i 群的所有樣本 index
        cluster_mask = kmeans.labels_ == i
        cluster_distances = distances[cluster_mask, i]
        
        # 如果該群樣本不足 5 個，就全拿
        top_k = min(5, sum(cluster_mask))
        
        # 找出距離最小的 top_k 個樣本的局部 index
        closest_local_indices = np.argsort(cluster_distances)[:top_k]
        
        # 轉回原始 cat_df 的 index
        actual_indices = np.where(cluster_mask)[0][closest_local_indices]
        
        for idx in actual_indices:
            row = cat_df.iloc[idx]
            gold_candidates_list.append({
                'text': row['name'],
                'proposed_category': cat,
                'cluster_id': i
            })

# 轉換成 DataFrame
df_gold_candidates = pd.DataFrame(gold_candidates_list)

print(f"\n✅ 採樣完成！共萃取出 {len(df_gold_candidates)} 筆高品質候選樣本。")
# 儲存成 CSV 讓您可以人工快速看過
df_gold_candidates.to_csv('gold_candidates_review.csv', index=False, encoding='utf-8-sig')

# 預覽一下借貸融資的前 5 筆
df_gold_candidates[df_gold_candidates['proposed_category'] == '借貸融資'].head()

啟動黃金樣本自動採樣 (K-Means Centroid Sampling)...
處理類別: 黃特種行業
處理類別: 詐騙高風險招募
處理類別: 借貸融資
處理類別: 博弈

✅ 採樣完成！共萃取出 545 筆高品質候選樣本。


,text,proposed_category,cluster_id
247,中彰二胎貸款 土地 房屋 持分 借款,借貸融資,0
248,房屋二胎貸款、土地持分貸款、持分房屋貸款、公同共有持分買賣、張代書,借貸融資,0
249,龐德隆不動產融資公司《房屋土地一二胎案件交流、房屋土地一二胎貸款、房屋土地轉貸增貸降息》,借貸融資,0
250,正順資產鈔好貸 專辦銀行房屋貸款/持分不動產收購或借貸/房屋土地二三胎 /不動產查封撤封/可...,借貸融資,0
251,鉅航不動產顧問公司-土地貸款房屋貸款,借貸融資,0


In [29]:
df_candidates['color_category'].value_counts()

color_category
黃特種行業      3182
借貸融資       2100
博弈          860
詐騙高風險招募     163
Name: count, dtype: int64